# 2. Speech Pipeline, LLM Gateway and Guardrails

This notebook combines the remaining three assignment topics in one short demonstration.

### Part A — Speech

`Audio → Whisper ASR → Qwen text LLM → gTTS audio`

### Part B — Gateway

A small Python gateway checks an API key, validates inputs, routes simple questions to a smaller model and detailed questions to Qwen, applies a basic per-user rate limit, and records request metadata.

### Part C — Guardrails

Python code checks tool permissions and tenant ownership before returning documents. It also validates model output.

> This is an ASR + text LLM + TTS pipeline, not a natively trained speech LLM. The gateway is an in-notebook demonstration, not a deployed HTTP service. The guardrails are basic application controls, not a complete security system.

The notebook automatically uses a GPU when one is available and otherwise runs on CPU. A GPU improves speed but is not required for this demonstration.

In [7]:
import sys
import subprocess
import importlib.util
import shutil
import time
import torch


def install_if_missing(import_name, package_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing missing package: {package_name}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_name
        ])
    else:
        print(f"Already installed: {package_name}")


install_if_missing("transformers", "transformers")
install_if_missing("gtts", "gTTS")
install_if_missing("sentencepiece", "sentencepiece")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Models will run on CPU; generation may be slower.")

if shutil.which("ffmpeg") is None:
    print("Note: ffmpeg was not found. Colab normally includes it; audio decoding may need ffmpeg.")

Already installed: transformers
Already installed: gTTS
Already installed: sentencepiece
Using device: cpu
No GPU detected. Models will run on CPU; generation may be slower.


## Part A — Build the speech pipeline

We use three separate components:

1. **Whisper Tiny** transcribes an audio recording.
2. **Qwen2.5-0.5B-Instruct** generates a text response.
3. **gTTS** converts that response into speech.

The code selects CUDA when available and CPU otherwise.

In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline,
)
from gtts import gTTS
from IPython.display import Audio, display

QWEN_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_NAME,
    torch_dtype=MODEL_DTYPE,
)
qwen_model.to(DEVICE)
qwen_model.eval()

print("Loaded text LLM:", QWEN_NAME)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded text LLM: Qwen/Qwen2.5-0.5B-Instruct


In [9]:
def ask_qwen(question, max_new_tokens=100):
    """Send a question to Qwen and return only the newly generated text."""

    messages = [
        {
            "role": "system",
            "content": "You are a helpful engineering assistant. Explain clearly and avoid unsupported claims."
        },
        {
            "role": "user",
            "content": question
        },
    ]
    inputs = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    # Slice the output to get only the newly generated tokens
    input_ids_length = inputs.input_ids.shape[1]
    new_tokens = output_ids[0, input_ids_length:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(ask_qwen("In one sentence, what is lift?"))

Lift refers to the upward force exerted by an object or air on its surface when it is in motion relative to the air.


In [10]:
asr_device = 0 if torch.cuda.is_available() else -1
asr_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

speech_to_text = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=asr_device,
    torch_dtype=asr_dtype,
)

print("Whisper Tiny is ready. Device:", "GPU" if asr_device == 0 else "CPU")

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

... (5 outputs left out)

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Whisper Tiny is ready. Device: CPU


### Provide an audio input

To make the demo easy to run, the next cell first creates a short sample audio file using gTTS and then transcribes it with Whisper.

If gTTS cannot reach the internet, the cell will let you upload your own short WAV or MP3 recording instead.

In [11]:
from pathlib import Path

# Keep True for a ready-made sample. Change to False to upload your own recording.
USE_SAMPLE_AUDIO = True
sample_sentence = "Please explain the difference between lift and drag on an aircraft."
audio_path = None

if USE_SAMPLE_AUDIO:
    try:
        gTTS(text=sample_sentence, lang="en").save("sample_input.mp3")
        audio_path = "sample_input.mp3"
        print("Created sample input audio.")
    except Exception as error:
        print("Could not create sample audio with gTTS:", str(error)[:250])

if audio_path is None:
    print("Upload a short WAV or MP3 recording.")
    try:
        from google.colab import files
        uploaded_files = files.upload()
        if uploaded_files:
            audio_path = next(iter(uploaded_files.keys()))
        else:
            raise ValueError("No audio file was uploaded.")
    except ImportError:
        raise RuntimeError("Please provide a WAV or MP3 file when running outside Colab.")

print("Audio file:", audio_path)
display(Audio(filename=audio_path))

Created sample input audio.
Audio file: sample_input.mp3


In [12]:
# Convert the audio into text.
asr_result = speech_to_text(audio_path)
transcribed_text = asr_result["text"].strip()

print("TRANSCRIBED TEXT:")
print(transcribed_text)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

TRANSCRIBED TEXT:
Please explain the difference between lift and drag on an aircraft.


## Part B — Build a simple LLM Gateway

A real gateway sits between applications and model deployments. It can provide authentication, model routing, rate limiting, retries, fallback and usage logging.

For this demonstration, we load two small models:

- **Fast route:** `google/flan-t5-small` for short, simple questions.
- **Detailed route:** the Qwen model already loaded above.

This makes model selection visible without API keys or separate cloud accounts.

In [13]:
# Load a small model for the fast route.
FAST_MODEL_NAME = "google/flan-t5-small"

fast_tokenizer = AutoTokenizer.from_pretrained(FAST_MODEL_NAME)
fast_model = AutoModelForSeq2SeqLM.from_pretrained(
    FAST_MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
)
fast_model.to(DEVICE)
fast_model.eval()

print("Loaded fast model:", FAST_MODEL_NAME)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded fast model: google/flan-t5-small


In [14]:
def ask_fast_model(question, max_new_tokens=60):
    """Generate a short answer using the smaller FLAN-T5 model."""
    prompt = "Answer clearly and briefly: " + question

    inputs = fast_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = fast_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return fast_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    ).strip()


print(ask_fast_model("What does ASR stand for?"))

ASR


### Add basic request validation and routing

The gateway below:

1. Checks a demo API key.
2. Rejects empty or very long questions.
3. Limits the number of requests per user during this runtime.
4. Selects a model based on the task type and question length.
5. Falls back to Qwen if the fast route fails.
6. Stores basic metadata (not the full user prompt) in a request log.

The API key and rate limit are in-memory teaching examples. Do not use them as production authentication or distributed rate limiting.

In [15]:
DEMO_API_KEY = "glassdata-demo-key"


def validate_input(question):
    """Check the request before sending it to a model."""
    if not isinstance(question, str):
        raise TypeError("The question must be text.")

    question = question.strip()

    if not question:
        raise ValueError("The question cannot be empty.")

    if len(question) > 2000:
        raise ValueError("The question is too long. Limit: 2000 characters.")

    return question


def validate_output(answer):
    """Reject empty or excessively long model responses."""
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("The model returned an empty response.")

    answer = answer.strip()
    if len(answer) > 4000:
        raise ValueError("The response is too long.")

    return answer


class SimpleLLMGateway:
    """A small in-process gateway for demonstrating common gateway logic."""

    def __init__(self, api_key, request_limit=5):
        self.api_key = api_key
        self.request_limit = request_limit
        self.request_counts = {}
        self.request_logs = []

    def choose_model(self, question, task_type):
        """Select a model route using a simple policy."""
        if task_type == "simple" and len(question) < 180:
            return "fast-flan-t5"
        return "detailed-qwen"

    def ask(self, api_key, user_id, question, task_type="simple"):
        # 1. Authenticate the caller.
        if api_key != self.api_key:
            raise PermissionError("Invalid API key.")

        # 2. Validate the incoming request.
        question = validate_input(question)

        # 3. Apply a basic per-user request limit.
        count = self.request_counts.get(user_id, 0)
        if count >= self.request_limit:
            raise RuntimeError("Rate limit reached for this demo user.")

        self.request_counts[user_id] = count + 1

        # 4. Choose the model route.
        selected_model = self.choose_model(question, task_type)
        start_time = time.time()

        # 5. Call the selected model and validate its response.
        # If the fast route fails or returns an empty response, try Qwen.
        try:
            if selected_model == "fast-flan-t5":
                answer = ask_fast_model(question)
            else:
                answer = ask_qwen(question)
            answer = validate_output(answer)
        except Exception as error:
            if selected_model == "fast-flan-t5":
                print("Fast route failed; switching to Qwen fallback:", str(error)[:200])
                selected_model = "detailed-qwen-fallback"
                answer = validate_output(ask_qwen(question))
            else:
                raise

        # 7. Record metadata without storing the full prompt.
        self.request_logs.append({
            "user_id": user_id,
            "model": selected_model,
            "latency_seconds": round(time.time() - start_time, 2),
            "status": "success",
        })

        return answer


gateway = SimpleLLMGateway(
    api_key=DEMO_API_KEY,
    request_limit=5,
)

simple_answer = gateway.ask(
    api_key=DEMO_API_KEY,
    user_id="demo-user-1",
    question="What is ASR?",
    task_type="simple",
)

print("Gateway response:", simple_answer)
print("Request log:", gateway.request_logs[-1])

Gateway response: ASR (disambiguation) ASR is a symphony of the symphony of the symphony of the symphony of the symphony of the symphony of
Request log: {'user_id': 'demo-user-1', 'model': 'fast-flan-t5', 'latency_seconds': 2.8, 'status': 'success'}


## Part C — Add guardrails around tools and private data

The model can suggest using a tool, but the application must check whether the authenticated user is allowed to use it.

We implement three checks in ordinary Python:

- **Tool allowlist:** only `search_docs` is available.
- **Permission check:** the user must have `docs:read` permission.
- **Tenant isolation:** documents are filtered by tenant before searching.

These controls are outside the language model, so the model cannot grant itself permissions.

In [16]:
# Example records belonging to two different teams.
DOCUMENTS = [
    {
        "tenant_id": "team-a",
        "title": "Lift equation",
        "text": "Lift equals one half times air density times velocity squared times wing area times lift coefficient."
    },
    {
        "tenant_id": "team-a",
        "title": "Wing aspect ratio",
        "text": "Aspect ratio is wingspan squared divided by wing planform area."
    },
    {
        "tenant_id": "team-b",
        "title": "Private project note",
        "text": "This document is private to team B."
    },
]

team_a_user = {
    "user_id": "user-101",
    "tenant_id": "team-a",
    "permissions": {"docs:read"},
}

team_b_user = {
    "user_id": "user-202",
    "tenant_id": "team-b",
    "permissions": {"docs:read"},
}


def search_documents(principal, query):
    """Search only documents owned by the authenticated user's tenant."""
    if "docs:read" not in principal["permissions"]:
        raise PermissionError("User does not have docs:read permission.")

    # Apply tenant filtering BEFORE searching document content.
    permitted_documents = [
        document for document in DOCUMENTS
        if document["tenant_id"] == principal["tenant_id"]
    ]

    query_words = [word.lower() for word in query.split() if len(word) > 2]
    matches = []

    for document in permitted_documents:
        searchable_text = (
            document["title"] + " " + document["text"]
        ).lower()

        if not query_words or any(word in searchable_text for word in query_words):
            matches.append(document)

    return matches


def execute_tool(principal, tool_name, arguments):
    """Validate a tool request before running it."""
    allowed_tools = {"search_docs"}

    if tool_name not in allowed_tools:
        raise PermissionError("This tool is not allowed.")

    if not isinstance(arguments, dict):
        raise ValueError("Tool arguments must be a dictionary.")

    query = arguments.get("query", "")
    if not isinstance(query, str) or len(query) > 200:
        raise ValueError("Invalid search query.")

    return search_documents(principal, query)


# Allowed request: team A searches its own documents.
team_a_results = execute_tool(
    principal=team_a_user,
    tool_name="search_docs",
    arguments={"query": "lift"},
)

print("Team A search results:")
for item in team_a_results:
    print("-", item["title"])

Team A search results:
- Lift equation


In [17]:
# Attempt 1: an unknown tool must be blocked.
try:
    execute_tool(
        principal=team_a_user,
        tool_name="delete_database",
        arguments={},
    )
except PermissionError as error:
    print("Blocked unknown tool:", error)

# Attempt 2: team A must not retrieve team B's private document.
team_a_private_search = execute_tool(
    principal=team_a_user,
    tool_name="search_docs",
    arguments={"query": "private project note"},
)

print("Team A results for team B's private note:", team_a_private_search)

# Attempt 3: a user without permission must be rejected.
user_without_permission = {
    "user_id": "guest-1",
    "tenant_id": "team-a",
    "permissions": set(),
}

try:
    execute_tool(
        principal=user_without_permission,
        tool_name="search_docs",
        arguments={"query": "lift"},
    )
except PermissionError as error:
    print("Blocked missing permission:", error)

Blocked unknown tool: This tool is not allowed.
Team A results for team B's private note: []
Blocked missing permission: User does not have docs:read permission.


## Part D — Connect the speech input to the gateway and speak the answer

We pass Whisper's transcription to the gateway. The gateway selects a model, validates the answer and returns text. gTTS then creates the spoken response.

In [18]:
assistant_answer = gateway.ask(
    api_key=DEMO_API_KEY,
    user_id="voice-user-1",
    question=transcribed_text,
    task_type="reasoning",
)

print("ASSISTANT TEXT RESPONSE:\n")
print(assistant_answer)

try:
    gTTS(text=assistant_answer, lang="en").save("assistant_reply.mp3")
    print("Generated spoken response:")
    display(Audio(filename="assistant_reply.mp3"))
except Exception as error:
    print("Text response succeeded, but online TTS failed:", str(error)[:250])

ASSISTANT TEXT RESPONSE:

Lift and drag are two important concepts in aerodynamics that describe how an object moves through air, particularly when it is flying or moving through the atmosphere.

**Drag (D):**
Drag is the force that opposes the motion of an object relative to the airflow. It arises from the interaction between the object's shape and the airflow. The more streamlined an object is, the less drag it experiences because there is less resistance due to turbulence and friction. Drag can be categorized into three types:
1.
Generated spoken response:


## Final summary and limitations

| Topic | What was demonstrated |
|---|---|
| Speech | Whisper ASR → Qwen text LLM → gTTS |
| LLM Gateway | API-key check, model routing, rate limit, fallback and request logs |
| Guardrails | Input/output validation, tool allowlist, permissions and tenant isolation |

### Limitations to state clearly

- This is a **cascaded speech pipeline**, not a native end-to-end speech LLM.
- The gateway is an in-process Python example, not a production HTTP gateway such as LiteLLM.
- The rate limit is stored in memory and resets when the runtime restarts. Production systems need shared storage such as Redis.
- The API key is a demonstration value. Production keys must be stored securely and rotated.
- The notebook does not claim to provide a complete prompt-injection detector or content moderation system.
- The tiny models and short examples are for demonstrating architecture, not measuring production accuracy.

### Useful references

- [Whisper Tiny model](https://huggingface.co/openai/whisper-tiny)
- [Qwen2.5-0.5B-Instruct model](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- [PEFT documentation](https://huggingface.co/docs/peft)
- [LiteLLM documentation](https://docs.litellm.ai/)
- [OWASP Top 10 for LLM Applications](https://genai.owasp.org/)